# 6.5 模型参数分析

本 notebook 介绍如何分析 PyTorch 模型的参数量、计算量和内存占用。

**学习目标：**
- 掌握手动计算模型参数量的方法
- 学会逐层分析参数分布
- 学会使用 torchinfo.summary() 进行自动化分析
- 能够对比不同模型的规模

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch 版本: {torch.__version__}")

## 1. 手动计算参数量

模型参数分为两类：
- **可训练参数（trainable）**: 通过反向传播更新的参数（权重、偏置）
- **不可训练参数（non-trainable）**: 固定不变的参数（如 BatchNorm 的 running_mean）

In [ ]:
class DemoModel(nn.Module):
    """用于参数分析的演示模型。"""
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32->16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16->8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8->4
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = DemoModel()
print("模型已创建")

In [ ]:
# 方法1: 计算总参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"总参数量:     {total_params:>10,}")
print(f"可训练参数:   {trainable_params:>10,}")
print(f"不可训练参数: {non_trainable_params:>10,}")
print(f"\n模型大小 (FP32): {total_params * 4 / 1024 / 1024:.2f} MB")
print(f"模型大小 (FP16): {total_params * 2 / 1024 / 1024:.2f} MB")

In [ ]:
# 方法2: 使用 named_parameters() 查看详细信息
print("所有参数:")
print(f"{'参数名':<30} {'形状':<20} {'参数量':>10} {'可训练':>6}")
print("-" * 70)

for name, param in model.named_parameters():
    print(f"{name:<30} {str(tuple(param.shape)):<20} {param.numel():>10,} "
          f"{'Yes' if param.requires_grad else 'No':>6}")

print("-" * 70)
print(f"{'总计':<30} {'':20} {total_params:>10,}")

## 2. 逐层参数分析

分析每一层的参数量和参数占比，了解模型参数的分布情况。

In [ ]:
def analyze_model_params(model: nn.Module) -> dict:
    """逐层分析模型参数。
    
    Args:
        model: PyTorch 模型
    
    Returns:
        包含各层参数信息的字典
    """
    total = sum(p.numel() for p in model.parameters())
    layer_info = []
    
    for name, module in model.named_children():
        params = sum(p.numel() for p in module.parameters())
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        if params > 0:  # 只显示有参数的层
            layer_info.append({
                "name": name,
                "type": module.__class__.__name__,
                "params": params,
                "trainable": trainable,
                "ratio": params / total * 100
            })
    
    return {"total": total, "layers": layer_info}

result = analyze_model_params(model)

print(f"{'层名':<10} {'类型':<15} {'参数量':>10} {'可训练':>10} {'占比':>8}")
print("=" * 60)

for info in result["layers"]:
    print(f"{info['name']:<10} {info['type']:<15} {info['params']:>10,} "
          f"{info['trainable']:>10,} {info['ratio']:>7.1f}%")

print("=" * 60)
print(f"{'总计':<10} {'':<15} {result['total']:>10,}")

print("\n观察: 全连接层通常占据最多参数")
print("这也是为什么现代架构倾向于使用全局平均池化替代全连接层")

## 3. 参数量计算公式

理解各层参数量的计算方式：

In [ ]:
# 手动验证参数量计算
print("各层参数量计算公式:")
print("=" * 60)

# Conv2d: out_channels * (in_channels * kH * kW + 1_bias)
conv1 = model.conv1
conv1_params = conv1.out_channels * (conv1.in_channels * conv1.kernel_size[0] * conv1.kernel_size[1])
conv1_bias = conv1.out_channels if conv1.bias is not None else 0
print(f"\nConv2d(3, 16, 3):")
print(f"  权重: {conv1.out_channels} x ({conv1.in_channels} x {conv1.kernel_size[0]} x {conv1.kernel_size[1]}) = {conv1_params}")
print(f"  偏置: {conv1_bias}")
print(f"  总计: {conv1_params + conv1_bias}")
print(f"  验证: {sum(p.numel() for p in conv1.parameters())}")

# BatchNorm2d: 2 * num_features (gamma + beta, running_mean + running_var 不算参数)
bn1 = model.bn1
bn1_params = sum(p.numel() for p in bn1.parameters())
bn1_buffers = sum(b.numel() for b in bn1.buffers())  # running_mean, running_var, num_batches_tracked
print(f"\nBatchNorm2d(16):")
print(f"  可训练参数 (gamma + beta): {bn1_params}")
print(f"  缓冲区 (running_mean + running_var + count): {bn1_buffers}")

# Linear: out_features * (in_features + 1_bias)
fc1 = model.fc1
fc1_params = fc1.out_features * fc1.in_features
fc1_bias = fc1.out_features if fc1.bias is not None else 0
print(f"\nLinear(1024, 256):")
print(f"  权重: {fc1.out_features} x {fc1.in_features} = {fc1_params}")
print(f"  偏置: {fc1_bias}")
print(f"  总计: {fc1_params + fc1_bias}")
print(f"  验证: {sum(p.numel() for p in fc1.parameters())}")

# 无参数的层
print(f"\nMaxPool2d, ReLU, Dropout: 0 个参数（无需学习）")

## 4. 使用 torchinfo.summary()

`torchinfo` 提供了类似 Keras `model.summary()` 的功能，可以一键生成模型摘要。

In [ ]:
# 安装 torchinfo（如果尚未安装）
# !pip install torchinfo

In [ ]:
try:
    from torchinfo import summary
    
    # 基本用法: 传入模型和输入尺寸
    print("基本摘要:")
    print("=" * 60)
    summary(model, input_size=(1, 3, 32, 32))

except ImportError:
    print("torchinfo 未安装，请运行: pip install torchinfo")
    print("以下为手动实现的简化版 summary")
    
    # 简化版 summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")
    print(f"Non-trainable params: {total_params - trainable_params:,}")
    print(f"Estimated Total Size (MB): {total_params * 4 / 1024 / 1024:.2f}")

In [ ]:
try:
    from torchinfo import summary
    
    # 高级用法: 自定义显示列
    print("自定义列的详细摘要:")
    print("=" * 60)
    summary(
        model,
        input_size=(1, 3, 32, 32),
        col_names=[
            "input_size",     # 输入尺寸
            "output_size",    # 输出尺寸
            "num_params",     # 参数量
            "kernel_size",    # 卷积核大小
            "mult_adds",      # 乘加次数 (FLOPs)
        ],
        depth=2,             # 显示深度
        verbose=1            # 详细程度: 0=简洁, 1=默认, 2=详细
    )

except ImportError:
    print("torchinfo 未安装，跳过此示例")

## 5. 对比不同模型的规模

定义几个不同规模的模型，对比它们的参数量和计算量。

In [ ]:
# 定义不同规模的模型

class TinyCNN(nn.Module):
    """极小 CNN: 约 5K 参数。"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 10)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class SmallCNN(nn.Module):
    """小型 CNN: 约 60K 参数。"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class MediumCNN(nn.Module):
    """中型 CNN: 约 300K 参数。"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

print("三个模型已定义")

In [ ]:
def compare_models(models: dict, input_size: tuple = (1, 3, 32, 32)):
    """对比多个模型的参数量和计算量。
    
    Args:
        models: {名称: 模型} 字典
        input_size: 输入尺寸
    """
    print(f"{'模型':<15} {'总参数':>12} {'可训练参数':>12} {'大小(MB)':>10} {'Conv参数':>12} {'FC参数':>12}")
    print("=" * 80)
    
    for name, model in models.items():
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        size_mb = total * 4 / 1024 / 1024
        
        # 统计 Conv 和 FC 参数
        conv_params = 0
        fc_params = 0
        for module in model.modules():
            if isinstance(module, nn.Conv2d):
                conv_params += sum(p.numel() for p in module.parameters())
            elif isinstance(module, nn.Linear):
                fc_params += sum(p.numel() for p in module.parameters())
        
        print(f"{name:<15} {total:>12,} {trainable:>12,} {size_mb:>10.3f} "
              f"{conv_params:>12,} {fc_params:>12,}")
    
    print("=" * 80)

models = {
    "TinyCNN": TinyCNN(),
    "SmallCNN": SmallCNN(),
    "MediumCNN": MediumCNN(),
    "DemoModel": DemoModel(),
}

compare_models(models)

In [ ]:
# 与经典模型对比（参数量参考值）
print("经典模型参数量参考:")
print("=" * 50)
print(f"{'模型':<20} {'参数量':>15} {'大小(MB)':>10}")
print("-" * 50)

reference_models = [
    ("LeNet-5", 60_000),
    ("AlexNet", 61_000_000),
    ("VGG-16", 138_000_000),
    ("ResNet-18", 11_700_000),
    ("ResNet-50", 25_600_000),
    ("ResNet-152", 60_200_000),
    ("MobileNetV2", 3_400_000),
    ("EfficientNet-B0", 5_300_000),
    ("ViT-Base", 86_000_000),
]

for name, params in reference_models:
    size_mb = params * 4 / 1024 / 1024
    print(f"{name:<20} {params:>15,} {size_mb:>10.1f}")

print("\n观察:")
print("- VGG-16 有 1.38 亿参数，大部分来自全连接层")
print("- ResNet-18 仅 1170 万参数，使用残差连接提升性能")
print("- MobileNetV2 仅 340 万参数，专为移动端设计")
print("- 参数量并不直接决定性能，架构设计更重要")

## 6. 计算 FLOPs（浮点运算量）

FLOPs（Floating Point Operations）衡量模型的计算量，是评估推理速度的重要指标。

In [ ]:
def estimate_flops(model: nn.Module, input_size: tuple = (1, 3, 32, 32)) -> dict:
    """估算模型的 FLOPs（简化版，仅计算 Conv 和 Linear）。
    
    Args:
        model: PyTorch 模型
        input_size: 输入尺寸 (B, C, H, W)
    
    Returns:
        各层 FLOPs 信息
    """
    flops_list = []
    hooks = []
    
    def conv_hook(module, input, output):
        # Conv2d FLOPs = 2 * Cout * Hout * Wout * Cin * kH * kW
        batch, out_c, out_h, out_w = output.shape
        in_c = module.in_channels
        kh, kw = module.kernel_size
        groups = module.groups
        flops = 2 * out_c * out_h * out_w * (in_c // groups) * kh * kw
        if module.bias is not None:
            flops += out_c * out_h * out_w
        flops_list.append((module.__class__.__name__, tuple(output.shape), flops))
    
    def linear_hook(module, input, output):
        # Linear FLOPs = 2 * in_features * out_features
        flops = 2 * module.in_features * module.out_features
        if module.bias is not None:
            flops += module.out_features
        flops_list.append((module.__class__.__name__, tuple(output.shape), flops))
    
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hooks.append(module.register_forward_hook(conv_hook))
        elif isinstance(module, nn.Linear):
            hooks.append(module.register_forward_hook(linear_hook))
    
    # 前向传播
    dummy_input = torch.randn(*input_size)
    with torch.no_grad():
        model(dummy_input)
    
    # 移除 hook
    for hook in hooks:
        hook.remove()
    
    return flops_list

# 分析 DemoModel 的 FLOPs
model = DemoModel()
flops_result = estimate_flops(model, input_size=(1, 3, 32, 32))

total_flops = 0
print(f"{'层类型':<12} {'输出形状':<20} {'FLOPs':>15} {'MFLOPs':>10}")
print("=" * 60)
for layer_type, output_shape, flops in flops_result:
    total_flops += flops
    print(f"{layer_type:<12} {str(output_shape):<20} {flops:>15,} {flops/1e6:>10.2f}")

print("=" * 60)
print(f"{'总计':<12} {'':20} {total_flops:>15,} {total_flops/1e6:>10.2f}")
print(f"\n总 FLOPs: {total_flops/1e6:.2f} MFLOPs")

## 7. 内存分析

In [ ]:
def estimate_memory(model: nn.Module, input_size: tuple = (1, 3, 32, 32)) -> dict:
    """估算模型的内存占用。
    
    Args:
        model: PyTorch 模型
        input_size: 输入尺寸
    
    Returns:
        内存占用信息字典
    """
    # 参数内存
    param_mem = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_mem = sum(b.numel() * b.element_size() for b in model.buffers())
    
    # 输入内存
    input_tensor = torch.randn(*input_size)
    input_mem = input_tensor.numel() * input_tensor.element_size()
    
    # 激活值内存（通过 hook 估算）
    activation_mem = 0
    hooks = []
    
    def hook_fn(module, input, output):
        nonlocal activation_mem
        if isinstance(output, torch.Tensor):
            activation_mem += output.numel() * output.element_size()
    
    for module in model.modules():
        if not isinstance(module, nn.Module) or module == model:
            continue
        hooks.append(module.register_forward_hook(hook_fn))
    
    with torch.no_grad():
        model(input_tensor)
    
    for hook in hooks:
        hook.remove()
    
    return {
        "params_mb": param_mem / 1024 / 1024,
        "buffers_mb": buffer_mem / 1024 / 1024,
        "input_mb": input_mem / 1024 / 1024,
        "activations_mb": activation_mem / 1024 / 1024,
        "total_mb": (param_mem + buffer_mem + input_mem + activation_mem) / 1024 / 1024
    }

model = DemoModel()
mem_info = estimate_memory(model)

print("内存占用估算 (推理时):")
print("=" * 40)
print(f"参数:     {mem_info['params_mb']:.3f} MB")
print(f"缓冲区:   {mem_info['buffers_mb']:.3f} MB")
print(f"输入:     {mem_info['input_mb']:.3f} MB")
print(f"激活值:   {mem_info['activations_mb']:.3f} MB")
print(f"-" * 40)
print(f"总计:     {mem_info['total_mb']:.3f} MB")
print(f"\n注意: 训练时还需要额外的梯度存储（约等于参数量）")
print(f"训练时估算: ~{mem_info['total_mb'] + mem_info['params_mb']:.3f} MB")

## 8. 总结

### 参数量计算

| 层类型 | 参数量公式 |
|--------|------------|
| Conv2d | out_ch * (in_ch * kH * kW) + out_ch(bias) |
| Linear | out_features * in_features + out_features(bias) |
| BatchNorm | 2 * num_features (gamma + beta) |
| MaxPool / ReLU / Dropout | 0 |

### 常用工具

```python
# 快速统计
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# torchinfo 详细报告
from torchinfo import summary
summary(model, input_size=(1, 3, 32, 32))
```

### 模型选择考虑因素
1. **参数量**: 影响模型大小和存储需求
2. **FLOPs**: 影响推理速度
3. **内存**: 影响 batch size 和 GPU 利用率
4. **性能**: 参数量大不一定性能好，架构设计更重要

---

## 练习题

**练习1（基础）：** 手动计算以下网络的参数量，并用代码验证：
```python
nn.Sequential(
    nn.Conv2d(3, 32, 5),
    nn.Conv2d(32, 64, 3),
    nn.Linear(64, 10)
)
```

**练习2（进阶）：** 定义三个不同的模型，使它们的参数量相近（约 100K），但使用不同的架构策略（深而窄 vs 浅而宽 vs 使用残差连接）。分析并对比它们的 FLOPs。

**练习3（挑战）：** 实现一个 `model_profiler` 函数，输入模型和输入尺寸，输出完整的分析报告，包括：逐层参数量、FLOPs、内存占用、参数占比饼图（文本形式）。